# DAT535 Data Exploration: Option 2 - Weather & Kolumbus Public Transit in Rogaland

This notebook tests data sources, APIs, historical coverage, update rates, volume, and suitability for the **DAT535 Medallion Architecture** project.

## 1. Project Concept
- **Bronze Layer**: Ingest real-time bus vehicle monitoring and estimated departure timetables via Entur SIRI API (`api.entur.io`) for Kolumbus (Rogaland public transport) alongside local weather station observations (Sola / Stavanger).
- **Silver Layer**: Calculate delay metrics (`aimedDepartureTime` vs `actualDepartureTime`), filter cancelled trips, clean invalid timestamps, map stop locations, and join hourly rainfall, wind speed, and snow with bus route delays.
- **Gold Layer**: Aggregated bus line punctuality report, delay severity under harsh weather conditions (wind > 15m/s or heavy rain), top delayed routes per weather condition.


In [1]:
import urllib.request
import json
import pandas as pd

# 1. Test Entur SIRI ET (Estimated Timetable / Realtime Delays) API
print('Fetching realtime estimated timetables from Entur API...')
url_siri = 'https://api.entur.io/realtime/v1/rest/et'
req = urllib.request.Request(url_siri, headers={'ET-Client-Name': 'UiS-DAT535-Explorer'})

try:
    with urllib.request.urlopen(req) as resp:
        content = resp.read().decode('utf-8')
        print(f'Successfully fetched live SIRI ET payload ({len(content)} characters)')
        # Show first 500 characters of raw XML/JSON stream
        print('Sample payload snippet:')
        print(content[:500])
except Exception as e:
    print('Error fetching SIRI ET:', e)


Fetching realtime estimated timetables from Entur API...


Successfully fetched live SIRI ET payload (257264746 characters)
Sample payload snippet:
<?xml version="1.0" encoding="UTF-8" standalone="yes"?><Siri xmlns="http://www.siri.org.uk/siri" xmlns:ns2="http://www.ifopt.org.uk/acsb" xmlns:ns3="http://www.ifopt.org.uk/ifopt" xmlns:ns4="http://datex2.eu/schema/2_0RC1/2_0" version="2.0"><ServiceDelivery><ResponseTimestamp>2026-10-09T15:02:07.385357449+02:00</ResponseTimestamp><ProducerRef>ENT</ProducerRef><EstimatedTimetableDelivery version="2.0"><ResponseTimestamp>2026-10-09T15:02:07.385380249+02:00</ResponseTimestamp><EstimatedJourneyVersi


In [2]:
# 2. Test Fetching Local High-Resolution Weather for Stavanger / Sola (Rogaland)
print('Fetching high-resolution weather data for Stavanger/Sola airport region...')
url_weather = 'https://archive-api.open-meteo.com/v1/archive?latitude=58.8767&longitude=5.6378&start_date=2024-01-01&end_date=2024-01-07&hourly=temperature_2m,precipitation,rain,snowfall,wind_speed_10m,wind_gusts_10m'

req = urllib.request.Request(url_weather, headers={'User-Agent': 'DAT535-Explorer'})
with urllib.request.urlopen(req) as resp:
    data = json.loads(resp.read().decode('utf-8'))['hourly']
    df_weather_rogaland = pd.DataFrame(data)

print(f'Rogaland weather rows for 1 week: {len(df_weather_rogaland)}')
display(df_weather_rogaland.head())


Fetching high-resolution weather data for Stavanger/Sola airport region...
Rogaland weather rows for 1 week: 168


,time,temperature_2m,precipitation,rain,snowfall,wind_speed_10m,wind_gusts_10m
0,2024-01-01T00:00,2.1,0.0,0.0,0.0,32.8,64.8
1,2024-01-01T01:00,2.1,0.0,0.0,0.0,38.7,76.3
2,2024-01-01T02:00,1.8,0.0,0.0,0.0,36.6,76.3
3,2024-01-01T03:00,1.8,0.0,0.0,0.0,34.5,72.7
4,2024-01-01T04:00,1.7,0.0,0.0,0.0,33.9,68.4


## 2. Technical Evaluation Summary (Option 2)

| Dimension | Details |
| :--- | :--- |
| **Data Availability** | Open and free. Entur API requires standard headers (`ET-Client-Name`). |
| **API Endpoint** | `api.entur.io/realtime/v1/rest/et` & `archive-api.open-meteo.com` |
| **History Range** | Weather: 1940 – Present. Transit Live API: Realtime / last 24h streaming (historical delay analysis requires saving continuous API snapshots over time or using Entur historical dumps). |
| **Update Frequency** | Live streaming (seconds to minutes). |
| **Data Volume** | Extremely high volume if capturing live transit streams (~10,000+ transit updates/day for Rogaland). |
| **Spark Suitability** | Great for real-time / streaming ingestion in Bronze and event-driven Silver cleaning. |
